# Session 3: Multi-Agent System (LangGraph)
**Goal**: Break monolithic agents into a TEAM of specialized agents with a conditional router and Human-In-The-Loop.

In [ ]:
import os
import sys
sys.path.insert(0, '..')
import utils.bootstrap

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing import Literal

from utils.tools import fetch_emails, schedule_meeting, draft_email_reply
from utils.llm_router import get_routed_llm

### Define Specialized Sub-Agents

In [ ]:
llm = get_routed_llm(role='master_model')
triage_agent = create_react_agent(llm, tools=[fetch_emails], prompt=SystemMessage(content="TRIAGE AGENT: Fetch emails and categorize as 'meeting', 'task', or 'info'."))
scheduler_agent = create_react_agent(llm, tools=[schedule_meeting], prompt=SystemMessage(content="SCHEDULER AGENT: Schedule the requested meetings."))
drafter_agent = create_react_agent(llm, tools=[draft_email_reply], prompt=SystemMessage(content="DRAFTER AGENT: Draft replies for tasks."))

### Define Graph Nodes & Router

In [ ]:
def triage_node(state: MessagesState):
    print("\n\ud83d\udd00 [TRIAGE] Analyzing...")
    return {"messages": triage_agent.invoke(state)["messages"]}

def router(state: MessagesState) -> Command[Literal["scheduler_node", "drafter_node", "__end__"]]:
    content = state["messages"][-1].content.lower()
    if "meeting" in content: return Command(goto="scheduler_node")
    elif "task" in content: return Command(goto="drafter_node")
    return Command(goto="__end__")

def scheduler_node(state: MessagesState):
    print("\n\ud83d\udcc5 [SCHEDULER] Booking...")
    return {"messages": scheduler_agent.invoke({"messages": state["messages"] + [HumanMessage(content="Schedule the meetings")]})["messages"]}

def drafter_node(state: MessagesState):
    print("\n\u270d\ufe0f [DRAFTER] Drafting...")
    return {"messages": drafter_agent.invoke({"messages": state["messages"] + [HumanMessage(content="Draft the replies")]})["messages"]}

def human_review_node(state: MessagesState):
    print("\n\ud83d\uded1 [HUMAN REVIEW] Requested action:")
    print(str(state["messages"][-1].content)[:200])
    decision = interrupt("Approve? (yes/no): ")
    return {"messages": state["messages"] + [AIMessage(content=f"Human decision: {decision}")]}

### Build & Run Graph

In [ ]:
builder = StateGraph(MessagesState)
builder.add_node("triage_node", triage_node)
builder.add_node("router", router)
builder.add_node("scheduler_node", scheduler_node)
builder.add_node("drafter_node", drafter_node)
builder.add_node("human_review_node", human_review_node)

builder.add_edge(START, "triage_node")
builder.add_edge("triage_node", "router")
builder.add_edge("scheduler_node", "human_review_node")
builder.add_edge("drafter_node", "human_review_node")
builder.add_edge("human_review_node", END)

app = builder.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "1"}}

# FIRST RUN (Will pause at interrupt)
print("\n\ud83d\ude80 Starting Graph...")
res = app.invoke({"messages": [HumanMessage(content="Analyze inbox.")]}, config=config)

### Resume Graph After Human Input

In [ ]:
# SECOND RUN (Resuming)
# Note: In Jupyter, standard `input()` might block if not careful, but works in cell mode.
human_input = input("\ud83d\uded1 Type 'yes' to approve: ")
res = app.invoke(Command(resume=human_input), config=config)
print("\n\u2728 FINAL:", res["messages"][-1].content)